# ZERO-DELAY CORRELATION MEASUREMENTS WITH PULSED SOURCES (PLAN 1) 
The LASER (PLD 800-D), is able to emmit pulses at the following frequencies: 
* Rep. Rate 1 = 1MHz    (One pulse each 1,000 ns)
* Rep. Rate 1 = 80MHz    (One pulse each 12.5 ns)

In [ ]:
# Usefull Functions used throuhgt the code. 
import numpy as np
import matplotlib.pyplot as plt
import TimeTagger as tt
import os

def measure_delay(tagger, ch_ref, ch_sig, duration_ps, bin_width=10, n_bins=2000):
    """
    Measures correlation between ch_ref and ch_sig.
    Returns the time axis, coincidence counts, and the delay at maximum coincidences.
    """
    # The Correlation class inherently measures Delta_t = t_sig - t_ref
    corr = tt.Correlation(tagger, ch_ref, ch_sig, binwidth=bin_width, n_bins=n_bins)
    
    tagger.sync()
    corr.startFor(duration_ps, clear=True)
    corr.waitUntilFinished()
    
    data = corr.getData()
    lags = corr.getIndex()
    
    # Identify the peak (the offset)
    max_idx = np.argmax(data)
    peak_delay = lags[max_idx]

    print(f'Peak Identified at {peak_delay} [ps] with {data[max_idx]}')
    
    return lags, data, peak_delay


def plot_delay(lags_s1,data_s1,peak_s1,lags_s2,data_s2,peak_s2,lags_12,data_12,peak_12, xlim_l=None, xlim_r=None):
    """
    Plots 3 graphs with the correlation measurements resulting from the measure_delay() function. 
    First graph is the delay od spad 1 (s1) vs laser sync signal. 
    Second graph is the delay between spad 2 (s2) and the laser sync signal. 
    Third graph is a sanity check that shows the delat between s1 and s2. 
    """
    #  Plotting
    fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=False)

    axs[0].plot(lags_s1, data_s1, color='blue') # Subplot 1
    axs[0].axvline(peak_s1, color='red', linestyle='--', label=f'Peak: {peak_s1} ps')
    axs[0].set_title("Sync vs Channel 1")
    axs[0].set_ylabel("Coincidences")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    axs[0].set_xlim(xlim_l,xlim_r)

    axs[1].plot(lags_s2, data_s2, color='green') # Subplot 2
    axs[1].axvline(peak_s2, color='red', linestyle='--', label=f'Peak: {peak_s2} ps')
    axs[1].set_title("Sync vs Channel 2")
    axs[1].set_ylabel("Coincidences")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    axs[1].set_xlim(xlim_l,xlim_r)

    axs[2].plot(lags_12, data_12, color='purple') # Subplot 3
    axs[2].axvline(peak_12, color='red', linestyle='--', label=f'Peak: {peak_12} ps')
    axs[2].set_title("Channel 1 vs Channel 2 (Sanity Check)")
    axs[2].set_xlabel("Delay (ps)")
    axs[2].set_ylabel("Coincidences")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)
    axs[2].set_xlim(xlim_l,xlim_r)

    plt.tight_layout()
    plt.show()

In [ ]:
# Standard naming parameters for the current measurement
source_type = "pulsed515nm"  # e.g., "pulsed405nm" or "cw660nm_diffuser"
source_freq = "20MHz"       # The laser frequency or diffuser speed

#  Data Saving
data_folder = f"../data/raw/correlation_2ch_zerodelay/{source_type}_{source_freq}"
os.makedirs(data_folder, exist_ok=True)

## Countrate Calibration
Calibrate countrates. Live monitor of counts-per-second.

* For LASER operating at certain frequency, attenuate to obtain 1% countrates at SPADS.
    * LASER at 1 MHz  1,000,000 cps -> 1\% =  1,000 cps. 
    * LASER at 80MHz 80,000,000 cps -> 1\% = 80,000 cps. 
* Uses ```tt.Countrate``` measurement class instance from swabian. 

In [ ]:
# Experimental Parameters
SPAD_A_CH = 1           # Port in Swabian for SPAD A
SPAD_B_CH = 2           # Port in Swabian for SPAD B
LASER_SYNC = 3
ADQ_TIME_PS = int(5e11) # In pico-seconds

# Countrate Experiment
tagger = tt.createTimeTagger()
tagger.setTriggerLevel(3, +0.5) #Laser sync trigger voltage
cr = tt.Countrate(tagger, channels=[SPAD_A_CH, SPAD_B_CH, LASER_SYNC]) # measurement class

# getSerial() is a nice trick to verify you are talking to the right machine
print(f"Connected to Swabian SN: {tagger.getSerial()}\nStarting Count Rate Monitor...")
print("(Press the 'Stop/Interrupt Kernel' button in Jupyter to exit)\n")

# Initialize list to store countrates
rates_A = []
rates_B = []
rates_LASER = []

try: 
    while True: 
        # clear=True guarantees we only look at the photons inside this specific 0.5s window
        cr.startFor(ADQ_TIME_PS, clear=True)    
        cr.waitUntilFinished()
        # Call measurement object > array 
        rates = cr.getData()
        # Separate rates for chanel A and B
        rate_A = rates[0]
        rate_B = rates[1]
        rate_LASER = rates[2]
        # Append to list
        rates_A.append(rate_A)
        rates_B.append(rate_B)
        rates_LASER.append(rate_LASER)
        # Live monitor 
        print(f"\r[LIVE] SPAD A: {rate_A:,.3f} cps  |  SPAD B: {rate_B:,.3f} cps | LASER Sync: {rate_LASER:,.3f} cps  ", end="", flush=True)

except KeyboardInterrupt:
    print("\n\nHalting Monitor...")
    rates_A_np = np.array(rates_A)
    rates_B_np = np.array(rates_B)
    rates_LASER_np = np.array(rates_LASER)
    if len(rates_A_np) > 0:
        mean_A = np.mean(rates_A_np)
        mean_B = np.mean(rates_B_np)
        mean_LASER = np.mean(rates_LASER)
        print(f"--- MEAN CPS OF {len(rates_A_np)} SAMPLES ---")
        print(f"Mean SPAD A: {mean_A:,.3f} cps")
        print(f"Mean SPAD B: {mean_B:,.3f} cps")
        print(f"Mean LASER: {mean_LASER:,.3f}cps")
        print("---------------------------------------")

finally: # Free hardware
    tt.freeTimeTagger(tagger)
    print("Hardware Released.")

## Signal Delay Calibration
Initializes tagger, channels and defines experimental parameters. Then it calls *measure_delay()* for each pair of channels: 
* First the delay od spad 1 (s1) vs laser sync signal. 
* Second the delay between spad 2 (s2) and the laser sync signal. 
* Third is a sanity check that shows the delay between s1 and s2.  
* Using ```measure_delay```, which in turn uses the ```tt.Correlation``` measurement class. 

In [ ]:
# Parameters
ch_1, ch_2, ch_sync = 1, 2, 3 

tagger = tt.createTimeTagger()  # Tagger object instance. 
tagger.setTriggerLevel(ch_sync, +0.5) # Adjust trigger level of laser sync

tagger.setInputDelay(ch_1, 0)   # Zero out all delays before calibrating
tagger.setInputDelay(ch_2, 0)
tagger.setInputDelay(ch_sync, 0)

calib_time = int(5e12) # pico-seconds
bin_w = 10             # ps resolution
bins = 4000            # bin_w * bins: total window. Ensure to catch the peak

# Execute Measurements
print("Measuring Sync vs Ch_1...")  
lags_s1, data_s1, peak_s1 = measure_delay(tagger, ch_sync, ch_1, calib_time, bin_w, bins) # Measurement A: Sync vs Ch1
print("Measuring Sync vs Ch_2...")
lags_s2, data_s2, peak_s2 = measure_delay(tagger, ch_sync, ch_2, calib_time, bin_w, bins) # Measurement B: Sync vs Ch2
print("Measuring Ch_1 vs Ch_2...")
lags_12, data_12, peak_12 = measure_delay(tagger, ch_1, ch_2, calib_time, bin_w, bins) # Measurement C: Ch1 vs Ch2 (The Sanity Check)

print("\n--- Calibration Results ---")  # Print Results: signal delays
print(f"Sync -> Ch1 Delay: {peak_s1} ps")
print(f"Sync -> Ch2 Delay: {peak_s2} ps")
print(f"Ch1  -> Ch2 Delay: {peak_12} ps")

#Visualize results
plot_delay(lags_s1,data_s1,peak_s1,lags_s2,data_s2,peak_s2,lags_12,data_12,peak_12,xlim_l = None, xlim_r=None)

#### Save Data (Uncalibrated)
Output: uncalib_Sync-Ch2_640nm_80MHz_res=10ps_run1.npz

In [ ]:
# Assume data_folder, source_type, source_freq are defined when saving data.

# Generate standard filenames
filename_S1 = f"{data_folder}/uncalib_Sync-Ch1_{source_type}_{source_freq}.npz"
filename_S2 = f"{data_folder}/uncalib_Sync-Ch2_{source_type}_{source_freq}.npz"
filename_12 = f"{data_folder}/uncalib_Ch1-Ch2_{source_type}_{source_freq}.npz"

# np.savez bundles multiple arrays into one file. 
np.savez(filename_S1, lags=lags_s1, counts=data_s1, peak=peak_s1)
np.savez(filename_S2, lags=lags_s2, counts=data_s2, peak=peak_s2)
np.savez(filename_12, lags=lags_12, counts=data_12, peak=peak_12)

print(f"Data safely bundled and saved to:")
print(f" - {filename_S1}")
print(f" - {filename_S2}")
print(f" - {filename_12}")

#### Apply delays to check coincidence peak is at 0. 
Re-run *### Measurement / Calibration # 1*  code with the applied delays. 

In [ ]:
# Assuming tagger, channels, trigger levels, calib_time, bin_w, bins defined in the first cell of calibration. 
 
tagger.setInputDelay(ch_1,  -10990 )   # Apply delays resulting from calibration
tagger.setInputDelay(ch_2,  -15210 )
tagger.setInputDelay(ch_sync, 0)

print(f"Running calibration ...")
# Execute Measurements
print("Measuring Sync vs Ch_1...")  
lags_s1, data_s1, peak_s1 = measure_delay(tagger, ch_sync, ch_1, calib_time, bin_w, bins) # Measurement A: Sync vs Ch1

print("Measuring Sync vs Ch_2...")
lags_s2, data_s2, peak_s2 = measure_delay(tagger, ch_sync, ch_2, calib_time, bin_w, bins) # Measurement B: Sync vs Ch2

print("Measuring Ch_1 vs Ch_2...")
lags_12, data_12, peak_12 = measure_delay(tagger, ch_1, ch_2, calib_time, bin_w, bins) # Measurement C: Ch1 vs Ch2 (The Sanity Check)

tt.freeTimeTagger(tagger)

print("\n--- Calibration Results ---")
print(f"Sync -> Ch1 Delay: {peak_s1} ps")
print(f"Sync -> Ch2 Delay: {peak_s2} ps")
print(f"Ch1  -> Ch2 Delay: {peak_12} ps")

expected_12 = peak_s2 - peak_s1
print(f"Sanity Check: Expected Ch1->Ch2 was {expected_12} ps. Measured was {peak_12} ps.")
print(f"Discrepancy: {abs(expected_12 - peak_12)} ps")

#Visualize results. 
#plot_delay(lags_s1,data_s1,peak_s1,lags_s2,data_s2,peak_s2,lags_12,data_12,peak_12,xlim_l = -16500, xlim_r=-3000)
plot_delay(lags_s1,data_s1,peak_s1,lags_s2,data_s2,peak_s2,lags_12,data_12,peak_12,xlim_l = None, xlim_r=None)


#### Save Data (Calibrated)
Output: calib_Sync-Ch2_640nm_80MHz_res=10ps_run1

In [ ]:
# Assume data_folder, source_type, source_freq are defined when saving data.

# Generate standard filenames
filename_S1 = f"{data_folder}/calib_Sync-Ch1_{source_type}_{source_freq}.npz"
filename_S2 = f"{data_folder}/calib_Sync-Ch2_{source_type}_{source_freq}.npz"
filename_12 = f"{data_folder}/calib_Ch1-Ch2_{source_type}_{source_freq}.npz"

# np.savez bundles multiple arrays into one file. 
np.savez(filename_S1, lags=lags_s1, counts=data_s1, peak=peak_s1)
np.savez(filename_S2, lags=lags_s2, counts=data_s2, peak=peak_s2)
np.savez(filename_12, lags=lags_12, counts=data_12, peak=peak_12)

print(f"Data safely bundled and saved to:")
print(f" - {filename_S1}")
print(f" - {filename_S2}")
print(f" - {filename_12}")

## Coincidences Measurement
At the heart of this measurement, three measurement classes are chained together: 
1. ```counter = tt.Counter(tagger, [ch_a, ch_b, ch_sync], binwidth=exposure_time, n_values=1) # Counter tracks total pulses and clicks```
2. ```coinc = tt.Coincidence(tagger, [ch_a, ch_b], coincidence_window) # Coincidence class for real-time hardware-level coincidence filtering```
3. ```coinc_counter = tt.Counter(tagger, [coinc.getChannel()], binwidth=exposure_time, n_values=1) # Count coincidences.```

In [ ]:
tt.freeTimeTagger(tagger)  # If needed
print("Hardware Released.")

In [ ]:
# Initialize Parameters
tagger = tt.createTimeTagger()

ch_a, ch_b, ch_sync = 1, 2, 3           # Define hardware ports
#tagger.setTriggerLevel(ch_a, 0.5)       # Voltage level for trigger. 
#tagger.setTriggerLevel(ch_b, 0.5)       # Voltage level for trigger. 
tagger.setTriggerLevel(ch_sync, +0.5)    # Voltage level for trigger. 

tagger.setInputDelay(ch_1,  -10990 )   # Apply delays resulting from calibration
tagger.setInputDelay(ch_2,  -15210 )
tagger.setInputDelay(ch_sync, 0)

exposure_time = 1e12 # picoseconds
jitter_ps = 24000    # 200 jitter in ps (reported by NVIZ)
coincidence_window = 2 * jitter_ps

# Data Acquisition
counter = tt.Counter(tagger, [ch_a, ch_b, ch_sync], binwidth=exposure_time, n_values=1) # Counter tracks total pulses and clicks
coinc = tt.Coincidence(tagger, [ch_a, ch_b], coincidence_window) # Coincidence class for real-time hardware-level coincidence filtering
coinc_counter = tt.Counter(tagger, [coinc.getChannel()], binwidth=exposure_time, n_values=1) # Count coincidences.

tagger.sync() # Clear FPGA pipeline

print(f"Adquiring data for {exposure_time/1e12:,} seconds...")
counter.startFor(exposure_time, clear=True)
coinc_counter.startFor(exposure_time, clear = True)
counter.waitUntilFinished()
coinc_counter.waitUntilFinished()

# Extract Results
counts = counter.getData()
events_a = counts[0][0]
events_b = counts[1][0]
total_pulses = counts[2][0]
coincidences = coinc_counter.getData()[0][0]

# 5. Math Analysis
p_a = events_a / total_pulses
p_b = events_b / total_pulses
p_ab = coincidences / total_pulses

g2_0 = p_ab / (p_a * p_b)

print(f"g2(0): {g2_0:.4f}")

### Save Data

In [ ]:
# Standard naming parameters for the current measurement
coinc_window = "24000ps"

# Generate the unique filename ID
filename_meas = f"{data_folder}/g2_meas_{source_type}_{source_freq}_{coinc_window}.npz"

# Bundle everything: raw arrays, extracted values, and calculated physics
np.savez(
    filename_meas,
    raw_counts=counts,                           # The full 3x1 array [Ch_A, Ch_B, Sync]
    raw_coincidences=coinc_counter.getData(),    # The 1x1 array of virtual coincidences
    events_a=events_a,
    events_b=events_b,
    total_pulses=total_pulses,
    coincidences=coincidences,
    p_a=p_a,
    p_b=p_b,
    p_ab=p_ab,
    g2_0=g2_0, 
    coinc_window_ps=coincidence_window
)# It is standard practice to save critical metadata inside the file too

print(f"\nMeasurement successfully bundled and saved to:")
print(f" -> {filename_meas}")

### Load Data
*3x1 array [Ch_A, Ch_B, Sync]*
* raw_counts

*1x1 array*
* raw_coincidences
* events_a
* events_b
* total_pulses
* coincidences
* p_a
* p_b
* p_ab
* g2_0
* coinc_window_ps

In [ ]:
# Read the saved data
saved_run = np.load("../data/raw/correlation_2ch_zerodelay/pulsed640nm_80MHz_100uW/g2_meas_pulsed640nm_80MHz_5ps.npz")

# Extract Values
recovered_g2 = saved_run['g2_0']        # g2(0) value
raw_counts = saved_run['raw_counts']    # Number of counts
raw_coincidences = saved_run["raw_coincidences"]    # Number of coincidences

print(f"g2(0) =  P_AB /  (P_A  P_B) =  {recovered_g2:.4f}")       
print(f"P_A = {saved_run['p_a']:,.3f} | P_B ={saved_run['p_b']:,.3f} | P_AB = {saved_run['p_ab']:,.3f}")
print(f"--------------------------------------------------------------------")
print(f" # of pulses: {raw_counts[2][0]:,}")
print(f"# coincidences: {raw_coincidences[0][0]:,}") 
print(f"# counts in A: {raw_counts[0][0]:,}={saved_run['events_a']:,} | counts in B: {raw_counts[1][0]:,} = {saved_run['events_b']:,}")
print(f"Coincidence Window: {saved_run['coinc_window_ps']:,} ps")

# DATA ANALYSIS

## Zero-delay correlation measurement for varius coincidence time tolerance. 

In [ ]:

# 4.1 Load all of g2_meas_pulsed640nm_1MHz_jitter=200ps_run1 for varying jitters: 
## 200 , 1000, 5000, 10000, 30000, 50000, 100000 
# 4.2 Load all of g2_meas_pulsed640nm_80MHz_jitter=200ps_run1 for varying jitters: 
## 200, 1000, 1500, 2000, 3000, 4000, 5000
## Why does higer jitter (that leads to coincidence timing tolerace) lead to higer g2 values? 
## What is the proper jitter? Why are results not converging? 

import numpy as np
import matplotlib.pyplot as plt

# Set data folders
data_folder1 = "../data/raw/correlation_2ch_zerodelay/pulsed640nm_1MHz_33uW/"
data_folder2 = "../data/raw/correlation_2ch_zerodelay/pulsed640nm_80MHz_100uW/"

windows_1MHz = [10,50,100,200,500,1000,5000,50000,100000,300000,400000,450000,500000,510000,550000,600000] # 1MHz_33uW
windows_80MHz = [10,20,30,40,50,60,70,80,90,100,120,150,200,300,400,500,600,700,1200,1500,2000,3000,4000,5000,6000,6100,6200,6300,6500,7000,10000]#,12500,15000] # 80MHz_100uW
#windows_1MHz = [5,10,50,100,150,200,300,400,500,600,700,800,900,1000,2000,6000,8000,10000,20000,40000,60000,80000,100000,200000,210000,220000,230000,240000,250000,300000,350000,400000,420000,460000,500000,501000,510000,700000] # run1_1MHz_3uW
#windows_80MHz =[5,50,100,500,1000,1100,5000,6000,6100,6200,6300,7000,] # Incomplete list

g2_1MHz = []
g2_80MHz = []

# Load 1 MHz Data
for w in windows_1MHz:
    filename = f"{data_folder1}/g2_meas_pulsed640nm_1MHz_{w}ps.npz"
    with np.load(filename) as data:
        g2_1MHz.append(data['g2_0']) 

# Load 80 MHz Data
for w in windows_80MHz:
    filename = f"{data_folder2}/g2_meas_pulsed640nm_80MHz_{w}ps.npz"
    with np.load(filename) as data:
        g2_80MHz.append(data['g2_0'])

# Plot
windows_1MHz = np.array(windows_1MHz)
windows_80MHz = np.array(windows_80MHz)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1 MHz Plot
ax1.plot(windows_1MHz*2, g2_1MHz, marker='o', linestyle='-', color='blue', linewidth=2)
ax1.set_title("1 MHz Source: $g^{(2)}(0)$ vs. Coincidence Window")
ax1.set_xlabel("Coincidence Window: $\Delta t_{coincidence}$")
ax1.set_ylabel("Calculated $g^{(2)}(0)$")
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log') # Log scale helps visualize the massive jump to 100,000 ps
ax1.axvspan(400, 1000000000000/1000000, color='green', alpha=0.1, label='Target Window (~2x FWHM)')
ax1.legend()

# 80 MHz Plot
ax2.plot(windows_80MHz*2, g2_80MHz, marker='s', linestyle='-', color='red', linewidth=2)
ax2.set_title("80 MHz Source: $g^{(2)}(0)$ vs. Coincidence Window")
ax2.set_xlabel("Coincidence Window: $\Delta t_{coincidence}$")
ax2.set_ylabel("Calculated $g^{(2)}(0)$")
ax2.grid(True, alpha=0.3)
ax2.axvspan(400, 1000000000000/80000000, color='green', alpha=0.1, label='Target Window (~2x FWHM)')
ax2.legend()

plt.tight_layout()
plt.show()

## Calibration Results

In [ ]:
# Locate Correlation Data
data_folder = "../data/raw/correlation_2ch_zerodelay/pulsed640nm_80MHz_100uW/"
laser_wl = "pulsed640nm"     
laser_freq = "80MHz" 
run_id = "res=10ps_run1"

# Load files
uc_filename_S1 = f"{data_folder}/uncalib_Sync-Ch1_{laser_wl}_{laser_freq}.npz" # uncorrelated
uc_filename_S2 = f"{data_folder}/uncalib_Sync-Ch2_{laser_wl}_{laser_freq}.npz"
uc_filename_12 = f"{data_folder}/uncalib_Ch1-Ch2_{laser_wl}_{laser_freq}.npz"
c_filename_S1 = f"{data_folder}/calib_Sync-Ch1_{laser_wl}_{laser_freq}.npz"  # correlated
c_filename_S2 = f"{data_folder}/calib_Sync-Ch2_{laser_wl}_{laser_freq}.npz"
c_filename_12 = f"{data_folder}/calib_Ch1-Ch2_{laser_wl}_{laser_freq}.npz"

# Load the data for Sync vs Channel 1
with np.load(uc_filename_S1) as data:
    uc_lags_s1 = data['lags']
    uc_data_s1 = data['counts']
    uc_peak_s1 = data['peak']
# Load the data for Sync vs Channel 2
with np.load(uc_filename_S2) as data:
    uc_lags_s2 = data['lags']
    uc_data_s2 = data['counts']
    uc_peak_s2 = data['peak']
# Load the data for Channel 1 vs Channel 2
with np.load(uc_filename_12) as data:
    uc_lags_12 = data['lags']
    uc_data_12 = data['counts']
    uc_peak_12 = data['peak']

# Load the data for Sync vs Channel 1
with np.load(c_filename_S1) as data:
    c_lags_s1 = data['lags']
    c_data_s1 = data['counts']
    c_peak_s1 = data['peak']

# Load the data for Sync vs Channel 2
with np.load(c_filename_S2) as data:
    c_lags_s2 = data['lags']
    c_data_s2 = data['counts']
    c_peak_s2 = data['peak']

# Load the data for Channel 1 vs Channel 2
with np.load(c_filename_12) as data:
    c_lags_12 = data['lags']
    c_data_12 = data['counts']
    c_peak_12 = data['peak']

# Visualize Load
print("--- Loaded Calibration Results ---")
print(f"Sync -> Ch1 Peak Delay: {uc_peak_s1} ps")
print(f"Sync -> Ch2 Peak Delay: {uc_peak_s2} ps")
print(f"Ch1  -> Ch2 Peak Delay: {uc_peak_12} ps")
plot_delay(uc_lags_s1,uc_data_s1,uc_peak_s1,uc_lags_s2,uc_data_s2,uc_peak_s2,uc_lags_12,uc_data_12,uc_peak_12,xlim_l = None, xlim_r=None)
plot_delay(c_lags_s1,c_data_s1,c_peak_s1,c_lags_s2,c_data_s2,c_peak_s2,c_lags_12,c_data_12,c_peak_12,xlim_l = -5000, xlim_r=5000)


## Bonus: Jitter

In [ ]:
# Calculate jitter with normalization and FWHM
def full_width_half_max(lags, counts):
    """
    Calculates the FWHM (Jitter) of a TCSPC histogram.
    Returns the FWHM, the left time, the right time, and the normalized array.
    """
    # 1. Normalize the data (0.0 to 1.0)
    baseline = np.mean(counts[:50]) #last 50 values
    norm_counts = (counts - baseline) / (np.max(counts) - baseline)
    
    # 2. Find the index of the absolute maximum
    max_idx = np.argmax(norm_counts)
    
    # 3. Split the data in two
    left_counts = norm_counts[:max_idx]
    left_lags = lags[:max_idx]
    
    right_counts = norm_counts[max_idx:]
    right_lags = lags[max_idx:]
    
    # 4. Find the half-max indices
    # np.abs(...) - 0.5 finds the distance from exactly 0.5
    # np.argmin finds the index of the value closest to 0.5
    idx_left = np.argmin(np.abs(left_counts - 0.5))
    idx_right = np.argmin(np.abs(right_counts - 0.5))
    
    # Linear Interpolation for sub-bin accuracy
    # Instead of just picking the closest chunky bin (which limits you to 5ps or 10ps resolution),
    # we draw a straight line between the points crossing 0.5 to find the exact sub-picosecond crossing.
    
    # Left interpolation
    x1, x2 = left_lags[idx_left], left_lags[idx_left + 1]
    y1, y2 = left_counts[idx_left], left_counts[idx_left + 1]
    t_left = x1 + (0.5 - y1) * (x2 - x1) / (y2 - y1)
    
    # Right interpolation
    x3, x4 = right_lags[idx_right - 1], right_lags[idx_right]
    y3, y4 = right_counts[idx_right - 1], right_counts[idx_right]
    t_right = x3 + (0.5 - y3) * (x4 - x3) / (y4 - y3)
    
    # 5. Calculate FWHM
    fwhm = t_right - t_left
    
    return fwhm, t_left, t_right, norm_counts


# Extract FWHM
FWHM_S1, t_left_S1, t_right_S1, norm_S1 = full_width_half_max(uc_lags_s1, uc_data_s1)
FWHM_S2, t_left_S2, t_right_S2, norm_S2 = full_width_half_max(uc_lags_s2, uc_data_s2)

print("=== JITTER RESULTS ===")
print(f"SPAD 1 FWHM: {FWHM_S1:.2f} ps | Jitter: {FWHM_S1/(2*np.sqrt(2*np.log(2))):.2f} ps")
print(f"SPAD 2 FWHM: {FWHM_S2:.2f} ps | Jitter: {FWHM_S2/(2*np.sqrt(2*np.log(2))):.2f} ps")

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Bundle data to iterate easily and keep the code DRY (Don't Repeat Yourself)
spad_data = [
    ("SPAD 1", uc_lags_s1, norm_S1, FWHM_S1, t_left_S1, t_right_S1, 'blue'),
    ("SPAD 2", uc_lags_s2, norm_S2, FWHM_S2, t_left_S2, t_right_S2, 'purple')
]

for col, (name, lags, norm, fwhm, t_left, t_right, color) in enumerate(spad_data):
    
    # -- Top Row: Linear Scale --
    ax_lin = axes[0, col]
    ax_lin.plot(lags, norm, drawstyle='steps-mid', color=color, alpha=0.8, label='Normalized Data')
    ax_lin.hlines(y=0.5, xmin=t_left, xmax=t_right, color='red', linewidth=2, label=f'FWHM = {fwhm:.1f} ps')
    ax_lin.axvline(t_left, color='red', linestyle='--', alpha=0.4)
    ax_lin.axvline(t_right, color='red', linestyle='--', alpha=0.4)
    
    ax_lin.set_xlim(t_left - 1500, t_right + 1500) # Zoom window
    ax_lin.set_ylim(-0.05, 1.1)
    ax_lin.set_title(f"{name} + LASER + Swabian Jitter (Linear Scale)")
    ax_lin.set_xlabel("Delay (ps)")
    ax_lin.set_ylabel("Normalized Counts")
    ax_lin.legend(loc="upper right")
    ax_lin.grid(alpha=0.3)

    # -- Bottom Row: Log Scale --
    ax_log = axes[1, col]
    ax_log.plot(lags, norm, drawstyle='steps-mid', color=color, alpha=0.8, label='Normalized Data')
    ax_log.hlines(y=0.5, xmin=t_left, xmax=t_right, color='red', linewidth=2)
    ax_log.axvline(t_left, color='red', linestyle='--', alpha=0.4)
    ax_log.axvline(t_right, color='red', linestyle='--', alpha=0.4)
    
    ax_log.set_xlim(t_left - 1500, t_right + 1500)
    ax_log.set_ylim(1e-3, 1.5) 
    ax_log.set_yscale('log')
    ax_log.set_title(f"{name} FWHM (Semi-Log Scale)")
    ax_log.set_xlabel("Delay (ps)")
    ax_log.set_ylabel("Normalized Counts (Log)")
    ax_log.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# Run this analisis (1 & 2) for: 
## calib_Sync-Ch1_640nm_1MHz_res=5ps_run1   &   calib_Sync-Ch1_640nm_1MHz_res=10ps_run2
## calib_Sync-Ch2_640nm_1MHz_res=5ps_run1   &   calib_Sync-Ch2_640nm_1MHz_res=10ps_run2

# Run the same analysis for: (tricky cuss multiple peaks in data)
## calib_Sync-Ch1_640nm_80MHz_res=5ps_run1   &   calib_Sync-Ch1_640nm_80MHz_res=10ps_run2
## calib_Sync-Ch2_640nm_80MHz_res=5ps_run1   &   calib_Sync-Ch2_640nm_80MHz_res=10ps_run2

# How does a lower res affect jitter measurement? 
# How does a higer laser frequency affect jitter measurement?
